# 05. Query Builder

This notebook documents the **query builder** component of our RAG pipeline.

`build_query(...)` takes:
- a list of ingredients (user pantry),
- an optional diet (e.g. *vegetarian*),
- an optional cuisine (e.g. *Italian*),

and produces a **natural language query string** that we use to:

1. Retrieve similar recipes in the vector space.
2. Provide additional context to the LLM prompt.

In [7]:
import os
import sys
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("Project root added: ", project_root)

Project root added:  /Users/biancaleoveanu/CookMate-Recipe-Generator


In [8]:
from rag_pipeline.query_builder import build_query

## 1. Simple examples

We start with a few simple inputs and inspect the resulting query string.

In [10]:
examples = [
    (["chicken"], None, None),
    (["chicken", "rice", "onion"], "vegetarian", None),
    (["shrimp", "garlic"], None, "Asian"),
    (["pasta", "tomato", "basil"], "vegetarian", "Italian"),
]

for ingredients, diet, cuisine in examples:
    print("=" * 80)
    print("Inputs:")
    print("  ingredients:", ingredients)
    print("  diet:", diet)
    print("  cuisine:", cuisine)
    print("-" * 80)
    print("Query:")
    print(build_query(ingredients=ingredients, diet=diet, cuisine=cuisine))
    print()

Inputs:
  ingredients: ['chicken']
  diet: None
  cuisine: None
--------------------------------------------------------------------------------
Query:
chicken

Inputs:
  ingredients: ['chicken', 'rice', 'onion']
  diet: vegetarian
  cuisine: None
--------------------------------------------------------------------------------
Query:
vegetarian chicken rice onion

Inputs:
  ingredients: ['shrimp', 'garlic']
  diet: None
  cuisine: Asian
--------------------------------------------------------------------------------
Query:
Asian shrimp garlic

Inputs:
  ingredients: ['pasta', 'tomato', 'basil']
  diet: vegetarian
  cuisine: Italian
--------------------------------------------------------------------------------
Query:
vegetarian Italian pasta tomato basil



## 2. Edge cases

We check behavior when:
- no ingredients,
- diet or cuisine missing,
- ingredients passed as a comma-separated string.

In [11]:
edge_cases = [
    ([], None, None),
    (["rice"], None, "Mexican"),
    ("tomato, onion, garlic", "vegan", None),
]

for ingredients, diet, cuisine in edge_cases:
    print("=" * 80)
    print("Inputs (edge case):")
    print("  ingredients:", ingredients)
    print("  diet:", diet)
    print("  cuisine:", cuisine)
    print("-" * 80)
    print("Query:")
    print(build_query(ingredients=ingredients, diet=diet, cuisine=cuisine))
    print()

Inputs (edge case):
  ingredients: []
  diet: None
  cuisine: None
--------------------------------------------------------------------------------
Query:


Inputs (edge case):
  ingredients: ['rice']
  diet: None
  cuisine: Mexican
--------------------------------------------------------------------------------
Query:
Mexican rice

Inputs (edge case):
  ingredients: tomato, onion, garlic
  diet: vegan
  cuisine: None
--------------------------------------------------------------------------------
Query:
vegan t o m a t o ,   o n i o n ,   g a r l i c



## 3. Integration with search

Finally, we show how the query builder conceptually connects to the retrieval step.
(Implementation of retrieval is in `rag_pipeline.search`, demonstrated in notebook 04.)

Here we only demonstrate the **string** we would send to the encoder.

In [12]:
ingredients = ["chicken", "rice", "onion"]
diet = "vegetarian"
cuisine = "Italian"

query_str = build_query(ingredients=ingredients, diet=diet, cuisine=cuisine)
print(query_str)

vegetarian Italian chicken rice onion
